# 5 · unspecified-goal kv-transfer — algorithm swap with an un-named sub-goal

The usual kv-transfer specifies every sub-goal in the task description. Here the instruction names **only the algorithm**
(`Implement breadth-first search …`) and the function is called `func`. The sub-goal at the pivot (e.g. *make a queue* vs
*make a stack*) is never written down; the model has to derive it from the algorithm's name.

```
p_src: "Implement A"  + shared prefill  ->  sub-goal of A      (e.g. BFS -> queue)
p_dst: "Implement B"  + shared prefill  ->  sub-goal of B      (e.g. DFS -> stack)
```
**Intervention** (= the paper's kv-transfer config): cache SRC's key+value over the algorithm-name span on the goal-250
heads and transplant it onto DST's span (`pad` alignment, GQA-corrected headwise routing — only the patch heads read the
source span; `QUERY_POSITIONS="last"` restricts that to the last query of each forward, so the pre-pivot DST
latents are untouched — under `"all"` every query after the span reads it). α boosts the patch heads' OV write at `-1` of every forward:
the pivot on the prefill pass, then each generated token (exactly as `run_kv_transfer_intervention`).

**Readouts** — GoalScope at the pivot (last prompt token) plus any prompt positions you list in `READ_AT`, on
clean SRC / DST / patched DST. The patched readout runs under the kv-transplant (+ α at the pivot only — the earlier
prompt tokens were never boosted during generation either). Readout positions must lie inside the prompt.


In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")   # check nvidia-smi first (shared machine)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, re
from contextlib import ExitStack
import torch
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
from src.diff.functional import (align_kv_spans, boost_head_ov, cache_src_proj, get_prompt, greedy,
                                 headwise_kv_patch, heads_per_layer, token_span)
from src.goals.goalscope import apply_goalscope
from src.tokens import prepare_input
from src.utils import env_utils

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich import print as rich_print
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]
print(f"{len(GOAL_HEADS_250)} goal heads")

## Knobs

In [ ]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0                 # OV-write boost of PATCH_HEADS at -1 (pivot / each generated token) on the PATCHED run; 1.0 = off
ALIGN_KNOB = "pad"          # "pad" = newline-pad the shorter span so the dest span is fully overwritten | "right"
QUERY_POSITIONS = "last"    # DST queries of PATCH_HEADS that read the transplanted span: "last" (pivot, then each
                            #   generated token; pre-pivot latents untouched) | "all" (every query after the span)
SPAN = "name"               # transplanted span: "name" (just the algorithm phrase) | "instruction" (the whole user turn)
PATCH_HEADS = GOAL_HEADS_250
READOUT_HEADS = GOAL_HEADS_250
MAX_NEW = 64                # greedy tokens per generation
RO_MAX_NEW = 48             # tokens per GoalScope verbalization
#------------------------------------------------------------------------------------------------------

## Contrastive pairs — pick one

Each pair: two algorithms with the same signature and a shared code prefix that is natural for both; the pivot is the
last prefill token (the indentation token before the first diverging line — Qwen tokenizes `\n    x` as `\n` … `'   '`, `' x'`).
`kw` regexes give a quick verdict on the first line the model writes. `FLIP=True` swaps SRC and DST.

In [ ]:
INSTRUCTION = "Implement {name} in Python. Name the function `func`."

PAIRS = {
    "bfs_dfs": dict(
        a=dict(id="bfs", name="breadth-first search graph traversal", kw=r"queue|deque|popleft"),
        b=dict(id="dfs", name="iterative depth-first search graph traversal", kw=r"stack"),
        prefill="<code>\ndef func(graph, start):\n    visited = set()\n   ",
    ),
    "bsearch_leftmost": dict(
        a=dict(id="binary_search", name="binary search", kw=r"==\s*target"),
        b=dict(id="leftmost", name="binary search that returns the index of the leftmost occurrence of target",
               kw=r"<\s*target|result|ans|res\b"),
        prefill=("<code>\ndef func(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n"
                 "        mid = (lo + hi) // 2\n       "),
    ),
    "merge_quick": dict(
        a=dict(id="merge_sort", name="merge sort", kw=r"mid|merge|//\s*2"),
        b=dict(id="quick_sort", name="quick sort", kw=r"pivot"),
        prefill="<code>\ndef func(arr):\n    if len(arr) <= 1:\n        return arr\n   ",
    ),
}

PAIR, FLIP = "bfs_dfs", False      # <- choose
# GoalScope positions besides the pivot (always included). Inside the prompt only.
#   int < 0  -> offset from the end of the prompt (-1 = pivot)      int >= 0 -> absolute index in the DST tokenization
#   str      -> last token in the prompt whose text (stripped) equals it, e.g. " visited" (skipped if absent)
READ_AT = [-2, " visited", " func"]

## Build SRC / DST prompts — task span (yellow), transplanted k/v window (orange), pivot (salmon)

In [ ]:
def question(alg):
    return INSTRUCTION.format(name=alg["name"])


def span_text(alg):
    return alg["name"] if SPAN == "name" else question(alg)


def build(pair_key, flip=False):
    pair = PAIRS[pair_key]
    src, dst = (pair["b"], pair["a"]) if flip else (pair["a"], pair["b"])
    src_prompt, dst_prompt = get_prompt(question(src), pair["prefill"]), get_prompt(question(dst), pair["prefill"])
    s0, t0 = token_span(src_prompt, span_text(src)), token_span(dst_prompt, span_text(dst))
    src_tok, dst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
        mt, src_prompt, dst_prompt, s0, t0, mode=ALIGN_KNOB)
    return dict(pair=pair_key, flip=flip, src=src, dst=dst, src_tok=src_tok, dst_tok=dst_tok,
                src_pos=src_pos, dst_pos=dst_pos, src_span=src_span, dst_span=dst_span)


def show_tokens(tok, title, task_span, kv_pos):
    ids = tok["input_ids"][0].tolist()
    task, kv, piv = set(range(*task_span)), set(kv_pos), len(ids) - 1
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid])
        style = ("black on salmon1" if i == piv else "black on dark_orange" if i in kv
                 else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{title}[/bold]  L={len(ids)}  pivot={mt.tokenizer.decode([ids[-1]])!r}",
                     title_align="left", border_style="grey42"))


def resolve_positions(tok, read_at):
    """READ_AT -> sorted absolute positions in `tok` (pivot always included)."""
    ids = tok["input_ids"][0].tolist()
    n, out = len(ids), {len(ids) - 1}
    for p in read_at:
        if isinstance(p, int):
            out.add(n + p if p < 0 else p)
        else:
            hits = [i for i, t in enumerate(ids) if mt.tokenizer.decode([t]).strip() == p.strip()]
            if not hits:
                print(f"READ_AT: token {p!r} not in this prompt — skipped")
                continue
            out.add(hits[-1])
    assert all(0 <= q < n for q in out), f"positions out of range: {sorted(out)} (L={n})"
    return sorted(out)


B = build(PAIR, FLIP)
for name, tok, span, pos in [("SRC", B["src_tok"], B["src_span"], B["src_pos"]),
                             ("DST", B["dst_tok"], B["dst_span"], B["dst_pos"])]:
    alg = B["src"] if name == "SRC" else B["dst"]
    show_tokens(tok, f"{name} · {alg['id']} · span={SPAN} {tuple(span)}", span, pos)
print(f"SRC = {B['src']['id']}   DST = {B['dst']['id']}   (transplant SRC -> DST)")

### Token picker — the prompt tail with absolute index and offset from end (for `READ_AT`)

In [ ]:
def token_picker(tok, tail=40):
    ids = tok["input_ids"][0].tolist()
    n = len(ids)
    t = Table(show_header=True, header_style="bold", box=None, pad_edge=False)
    for c in ("abs", "from end", "token"):
        t.add_column(c, justify="right" if c != "token" else "left")
    for i in range(max(0, n - tail), n):
        t.add_row(str(i), str(i - n), repr(mt.tokenizer.decode([ids[i]])))
    rich_print(t)


token_picker(B["dst_tok"])
print("READ_AT ->", resolve_positions(B["dst_tok"], READ_AT))

## Intervention — generations + GoalScope readouts

In [ ]:
def verdict(gen, b):
    first = next((ln for ln in gen.split("\n") if ln.strip()), "")
    hs, hd = bool(re.search(b["src"]["kw"], first)), bool(re.search(b["dst"]["kw"], first))
    return "SRC" if hs and not hd else "DST" if hd and not hs else "both" if hs else "neither"


def run(b, read_at=READ_AT):
    src_tok, dst_tok = b["src_tok"], b["dst_tok"]
    kv_layers = sorted(heads_per_layer(PATCH_HEADS, "k", layers={l for l, _ in PATCH_HEADS}))
    k_cache = cache_src_proj(src_tok, b["src_pos"], kv_layers, "k")
    v_cache = cache_src_proj(src_tok, b["src_pos"], kv_layers, "v")

    def patched():  # fresh single-use stack per forward / generation
        s = ExitStack()
        s.enter_context(headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, b["dst_pos"],
                                          query_positions=QUERY_POSITIONS))
        if ALPHA != 1.0:
            s.enter_context(boost_head_ov(mt, PATCH_HEADS, ALPHA, positions=(-1,)))
        return s

    def patched_iface(mt_, inputs, patches=None, model_kwargs=None):
        with patched():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))

    # generations: alpha at -1 of every forward = pivot on prefill, then each generated token
    gens = dict(src=greedy(src_tok, max_new=MAX_NEW), dst=greedy(dst_tok, max_new=MAX_NEW))
    with patched():
        gens["patched"] = greedy(dst_tok, max_new=MAX_NEW)

    # readout positions: resolved on DST, mapped to SRC by offset from the end (identical under pad)
    n_s, n_d = src_tok["input_ids"].shape[1], dst_tok["input_ids"].shape[1]
    pos_d = resolve_positions(dst_tok, read_at)
    pos_s = [p - n_d + n_s for p in pos_d]
    for ps, pd in zip(pos_s, pos_d):
        ts, td = (mt.tokenizer.decode([t["input_ids"][0, q]]) for t, q in ((src_tok, ps), (dst_tok, pd)))
        if ts != td:
            print(f"warning: SRC pos {ps} {ts!r} != DST pos {pd} {td!r} (position is inside the differing span?)")

    def ro(tok, pos, iface=None):
        res = apply_goalscope(prompt=tok, mt=mt, selected_heads=READOUT_HEADS, strategy="additive_norm",
                              alpha=0.1, steer_layer_range=(35, 46), batch_size=len(pos),
                              max_new_tokens=RO_MAX_NEW, goalscope_prompt=getattr(P, "gsp", None),
                              token_positions=pos, include_noncode=False, show=False, patch_interface=iface)
        return {r["pos"]: r["patched_gen"] for r in res}

    ros = dict(src=ro(src_tok, pos_s), dst=ro(dst_tok, pos_d), patched=ro(dst_tok, pos_d, iface=patched_iface))
    rows = [dict(pos=pd, from_end=pd - n_d, token=mt.tokenizer.decode([dst_tok["input_ids"][0, pd]]),
                 src=ros["src"][ps], dst=ros["dst"][pd], patched=ros["patched"][pd])
            for ps, pd in zip(pos_s, pos_d)]
    return dict(gens=gens, verdicts={k: verdict(g, b) for k, g in gens.items()}, rows=rows)


def show(b, out):
    head = (f"{b['pair']}{' (flipped)' if b['flip'] else ''} · SRC={b['src']['id']} -> DST={b['dst']['id']} · "
            f"α={ALPHA} · {ALIGN_KNOB} · span={SPAN} · q={QUERY_POSITIONS}")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        rich_print(Panel(Text(out["gens"][k]), title=f"[{col}]{k}[/{col}]  verdict={out['verdicts'][k]}",
                         title_align="left", border_style="grey42"))
    t = Table(title=head, show_lines=True)
    t.add_column("pos", justify="right"); t.add_column("from end", justify="right"); t.add_column("token")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        t.add_column(Text(f"{'clean SRC' if k == 'src' else 'DST' if k == 'dst' else 'patched DST'}", style=col), ratio=1)
    for r in out["rows"]:
        tok = Text(repr(r["token"]), style="bold salmon1" if r["from_end"] == -1 else "")
        t.add_row(str(r["pos"]), str(r["from_end"]), tok, *(Text(r[k].strip()) for k in ("src", "dst", "patched")))
    Console(width=200).print(t)


OUT = run(B)
show(B, OUT)

## Other pairs / directions (optional — each pair×direction is 3 generations + 3 readout batches)

In [ ]:
# for key in PAIRS:
#     for flip in (False, True):
#         b = build(key, flip)
#         show(b, run(b))